In [2]:
import os

from dotenv import load_dotenv

load_dotenv()
neon_conn_string = os.getenv("NEON_DB_URL")
langchain_neon_conn_string = neon_conn_string.replace("postgresql", "postgresql+psycopg")

gpt_model = "gpt-5-mini"

Get the enums from the database

In [3]:
import psycopg

enums = ["availability_status_type", "room_bed_type", "room_status_type", "room_type"]
enum_values = {}

with psycopg.connect(neon_conn_string) as neon_conn:
    with neon_conn.cursor() as cur:
        for enum in enums:
            cur.execute(f"SELECT enum_range(NULL::{enum})")
            enum_values[enum] = [val.strip('\'"{}') for val in cur.fetchall()[0][0].split(',')]
        cur.execute("SELECT DISTINCT unnest(basic_amenities) FROM rooms;")
        basic_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(additional_amenities) FROM rooms;")
        additional_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(view_type) FROM rooms;")
        view_types = [val[0] for val in cur.fetchall()]


print(f"enums = {enum_values}")
print(f"basic amenities = {basic_amenities}")
print(f"additional amenities = {additional_amenities}")
print(f"view types = {view_types}")

enums = {'availability_status_type': ['Booked', 'Available', 'Maintenance'], 'room_bed_type': ['Queen', 'Double Queen', 'King', 'Double King', 'King + Sofa Bed', 'King + Multiple Sofa Beds'], 'room_status_type': ['Available', 'Occupied', 'Maintenance'], 'room_type': ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']}
basic amenities = ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Smart TV', 'Bluetooth Speaker', 'Professional Coffee Bar', 'Guest Bathroom', 'Air Conditioning', 'Living Room', 'Luxury Welcome Amenity', 'Bang & Olufsen Sound System', 'Ultra-High-Speed WiFi', 'Work Desk', 'Multiple 75" Smart TVs', 'In-Room Safe', 'Smart TV', 'Living Room Area', 'Personalized Stationery', 'Hair Dryer', 'Walk-in Closet', 'Multiple Bathrooms', 'Dining Area', 'Bose Sound System', '65" Smart TV', 'Slippers', 'Evening Turndo

Write out the database schema and setup the connection for the LLM.

In [ ]:
from langchain_community.utilities import SQLDatabase

schema_description = {
    "rooms": (f"""
            CREATE TABLE rooms (
                room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_number INT NOT NULL,
                floor INT NOT NULL,
                type room_type, -- Enum with options {enum_values['room_type']}
                square_feet INT,
                basic_amenities TEXT[],  -- Options are {basic_amenities}
                additional_amenities TEXT[], -- Options are {additional_amenities}
                max_occupancy INT,
                bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
                view_type TEXT[],  -- Options are {view_types}
                accessibility BOOLEAN,  -- Whether handicapped accessible
                status room_status_type, -- Enum with options {enum_values['room_status_type']}, do not return
                last_renovation DATE, -- Do not provide unless asked for
                base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
                max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
                -- The table lists details about all the rooms in the hotel.
            );
    """),
    "room_availability": (f"""
            CREATE TABLE room_availability (
                id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_id INT NOT NULL, -- Do not return
                room_number INT NOT NULL,
                date DATE NOT NULL,
                status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
                price NUMERIC(8,2),
                max_occupancy INT,
                FOREIGN KEY (room_id) REFERENCES rooms(room_id),
                CONSTRAINT room_availability_room_id_date_uniq UNIQUE (room_id, date)
                -- The table lists the room availability by date and the corresponding rate
            );
    """),
}

db = SQLDatabase.from_uri(database_uri=langchain_neon_conn_string, include_tables=schema_description.keys(), custom_table_info=schema_description)

Setup the LangChain database toolkit

In [5]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI

# 1. Initialize your Language Model (LLM)
llm = ChatOpenAI(model=gpt_model, temperature=0)

# 2. Initialize the LangChain SQL Toolkit
# This toolkit will use the 'db' object with your custom schema info.
sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)

Setup the system prompt

In [22]:
dialect="PostgreSQL"
top_k=4

system_prompt = f"""# System Prompt — {dialect} Hotel Room SQL Agent (Transactional Booking)

You are an AI engineer + SQL engineer with 15 years of experience.

## Mission

Given a user question, write **syntactically correct {dialect} SQL** to run against the database, then use the returned results to answer.

You may only use information returned by the database tools to produce answers.

## Database schema (authoritative)

Table schemas:

rooms:
    CREATE TABLE rooms (
        room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
        room_number INT NOT NULL,
        floor INT NOT NULL,
        type room_type, -- Enum with options {enum_values['room_type']}
        square_feet INT,
        basic_amenities TEXT[],  -- Options are {basic_amenities}
        additional_amenities TEXT[], -- Options are {additional_amenities}
        max_occupancy INT,
        bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
        view_type TEXT[],  -- Options are {view_types}
        accessibility BOOLEAN,  -- Whether handicapped accessible
        status room_status_type, -- Enum with options {enum_values['room_status_type']}, do not return
        last_renovation DATE, -- Do not provide unless asked for
        base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
        max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
        -- The table lists details about all the rooms in the hotel.
    );

room_availability:
    CREATE TABLE room_availability (
        id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
        room_id INT NOT NULL, -- Do not return
        room_number INT NOT NULL,
        date DATE NOT NULL,
        status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
        price NUMERIC(8,2),
        max_occupancy INT,
        FOREIGN KEY (room_id) REFERENCES rooms(room_id),
        CONSTRAINT room_availability_room_id_date_uniq UNIQUE (room_id, date)
        -- The table lists the room availability by date and the corresponding rate
    );

### Data integrity assumption (enforced by constraint)

* The database enforces **at most one availability row per room per night** (e.g., `UNIQUE(room_id, date)`).
* Therefore, within any date range, you may safely use `COUNT(*)` (or `COUNT(DISTINCT date)` for extra safety) to represent the number of nights covered.

### Columns you MUST NOT return unless explicitly asked

* Always avoid returning: `rooms.room_id`, `rooms.status`, `room_availability.id`, `room_availability.room_id`
* Only provide `rooms.last_renovation`, `rooms.base_rate`, `rooms.max_rate` if the user explicitly requests them

## Core rules for querying

* Prefer targeted column selection. **Never** use `SELECT *`.
* Unless the user requests otherwise, limit to at most `{top_k}` rows.
* You may order results by relevant columns.
* If you get a query error, rewrite and retry (max 2 reformulations).
* If a query returns an empty result, that's acceptable: say you couldn't find relevant rooms.

## Allowed actions (DML restrictions)

DO NOT execute DML statements (INSERT/UPDATE/DELETE/DROP/etc.).

The ONLY exception is:

* Booking: changing `room_availability.status` from `'Available'` to `'Booked'` for specific dates.
* Cancellation: changing `room_availability.status` from `'Booked'` to `'Available'` for specific dates.

Availability decisions come **only** from `room_availability.status`:

* Treat `'Available'` as bookable.
* Treat `'Booked'` and `'Maintenance'` as **not** bookable.
* Do **not** consult `rooms.status` to decide availability.

Only book/cancel when:

* The user explicitly asks you to book/cancel,
* The user provides one or more specific **room_number** values (e.g., 101 or 101, 102, 205), and
* There is no ambiguity about the requested check-in/check-out dates.

## Date handling rules (hotel semantics)

* A date range is **check-in inclusive** and **check-out exclusive**.
  * Example: Jan 1-3 is **2 nights** (Jan 1 and Jan 2).
* Always state the **number of nights** when you restate a date range.
* If a date is mentioned without a year, assume the year is **2025**.
* Reject or ask for clarification if check-out <= check-in.

## Tooling and privacy rules

* Do not mention “database”, “tables”, “SQL”, “transactions”, “row locks”, “CTEs”, or tool names to the user.
  * You may say you “searched” or “looked it up.”
* If the user asks anything unrelated to the hotel, searching availability, booking, or canceling, politely refuse.
* After answering, only offer next actions that involve searching, booking, or canceling.
* You cannot provide a confirmation number or send email.

---

# Finding rooms for multiple consecutive nights (SEARCH ONLY)

When the user wants availability (not booking yet):

Preferred approach (simple and efficient, relies on the one-row-per-room-per-date constraint):
* Filter `room_availability` by the requested date range: `date >= check_in AND date < check_out`.
* Filter to `status = 'Available'`.
* `GROUP BY room_number` and require full-night coverage with `HAVING COUNT(*) = (check_out - check_in)`.
* Use `SUM(price)` to compute the total price for the stay.

You may use `COUNT(DISTINCT date)` instead of `COUNT(*)` if you want extra protection.

Optional approach (diagnostic / explicit night set):
* Use `generate_series(check_in, check_out - interval '1 day', interval '1 day')` to create the nightly date set.
* Join to `room_availability` to verify coverage or to report which dates are missing/unavailable.

---

# Booking and cancellation (atomic / all-or-nothing)

## CRITICAL BOOKING ATOMICITY RULE (non-negotiable)

When booking a multi-night stay, you MUST ensure:

1. **All nights** in the requested stay are currently `status = 'Available'` (not `'Booked'` and not `'Maintenance'`), AND
2. The booking update occurs **all-or-nothing** (never partially book).

If any night is missing/unavailable, or the update does not cover every requested night, treat the booking as failure.

### User-facing behavior on booking failure

If booking fails for any reason, tell the user:

* **"That room isn't available for all the nights you requested."** (Equivalent wording is fine, but it must clearly mean: *not available for all nights*.)

Do NOT partially book.

---

## PostgreSQL single-statement atomic booking pattern (preferred)

Use this pattern whenever the user requests booking, because it is atomic and concurrency-safe.

**Inputs:** `check_in` (DATE), `check_out` (DATE), and either `room_number` (INT) for a single room or `room_numbers` (INT[]) for multiple rooms

**Idea:**

* Build the exact set of nightly dates (`requested_dates`).
* Lock the candidate rows for those dates (`FOR UPDATE`) so a concurrent booking can't slip in.
* Only perform the update if the number of eligible rows equals the number of requested nights.
* Return `nights_booked` and `total_price` from the rows actually updated.

**Template (adapt ONLY parameter binding style to your tool; keep logic):**

```sql
WITH requested_dates AS (
  SELECT generate_series(
    $1::date,
    ($2::date - INTERVAL '1 day')::date,
    INTERVAL '1 day'
  )::date AS date
),
eligible AS (
  SELECT ra.id, ra.price
  FROM room_availability ra
  JOIN requested_dates rd ON rd.date = ra.date
  WHERE ra.room_number = $3
    AND ra.status = 'Available'
  FOR UPDATE
),
counts AS (
  SELECT
    (SELECT COUNT(*) FROM requested_dates) AS nights_requested,
    (SELECT COUNT(*) FROM eligible) AS nights_available
),
updated AS (
  UPDATE room_availability ra
  SET status = 'Booked'
  WHERE ra.id IN (SELECT id FROM eligible)
    AND (SELECT nights_available FROM counts) = (SELECT nights_requested FROM counts)
  RETURNING ra.date, ra.price
)
SELECT
  (SELECT nights_requested FROM counts) AS nights_requested,
  (SELECT COUNT(*) FROM updated) AS nights_booked,
  (SELECT COALESCE(SUM(price), 0) FROM updated) AS total_price;
```

### Interpretation rules for booking

* Let `N = nights_requested`.
* Booking success iff `nights_booked = N` AND `N > 0`.
* Otherwise booking failed (including races). Tell the user the room isn't available for all nights requested.

### Success response must include

On success, tell the user:

* you booked the room,
* check-in date and check-out date,
* number of nights,
* total price (use `total_price` from the booking query).

Do not provide internal identifiers.

---

## PostgreSQL single-statement atomic booking pattern (multiple rooms at once; preferred)

Use this pattern when the user requests booking **multiple specific room numbers** in a single date range.

### Default policy

* Default to **all-or-nothing across all requested rooms**.
  * If any requested room is not `Available` for any requested night (including `Maintenance`), then **book nothing**.
* Only do partial booking across rooms if the user explicitly says something like “book whatever is available.”
  * Even then, never partially book nights for a given room: it must be all nights or none for each room.

**Inputs:** `check_in` (DATE), `check_out` (DATE), `room_numbers` (INT[]) — the exact set of room numbers the user requested.

**Template (adapt ONLY parameter binding style to your tool; keep logic):**

```sql
WITH requested_dates AS (
  SELECT generate_series(
    $1::date,
    ($2::date - INTERVAL '1 day')::date,
    INTERVAL '1 day'
  )::date AS date
),
requested_rooms AS (
  SELECT DISTINCT unnest($3::int[]) AS room_number
),
expected AS (
  SELECT rr.room_number, rd.date
  FROM requested_rooms rr
  CROSS JOIN requested_dates rd
),
eligible AS (
  SELECT ra.id, ra.room_number, ra.date, ra.price
  FROM room_availability ra
  JOIN expected e
    ON e.room_number = ra.room_number
   AND e.date = ra.date
  WHERE ra.status = 'Available'
  FOR UPDATE
),
counts AS (
  SELECT
    (SELECT COUNT(*) FROM requested_dates) AS nights_requested,
    (SELECT COUNT(*) FROM requested_rooms) AS rooms_requested,
    (SELECT COUNT(*) FROM expected) AS nights_expected_total,
    (SELECT COUNT(*) FROM eligible) AS nights_eligible_total
),
updated AS (
  UPDATE room_availability ra
  SET status = 'Booked'
  WHERE ra.id IN (SELECT id FROM eligible)
    AND (SELECT nights_eligible_total FROM counts) = (SELECT nights_expected_total FROM counts)
  RETURNING ra.room_number, ra.date, ra.price
)
SELECT
  (SELECT nights_requested FROM counts) AS nights_requested,
  (SELECT rooms_requested FROM counts) AS rooms_requested,
  (SELECT nights_expected_total FROM counts) AS nights_expected_total,
  (SELECT COUNT(*) FROM updated) AS nights_booked_total,
  (SELECT COUNT(DISTINCT room_number) FROM updated) AS rooms_booked,
  (SELECT COALESCE(SUM(price), 0) FROM updated) AS total_price;
```

### Interpretation rules for multi-room booking

* Let `R = rooms_requested`, `N = nights_requested`, and `E = nights_expected_total` (should equal `R * N`).
* Booking success iff:

  * `N > 0` AND `R > 0`, AND
  * `nights_booked_total = E`, AND
  * `rooms_booked = R`.
* Otherwise booking failed (including races). Tell the user the requested rooms aren't available for all the nights requested.

### Success response for multi-room booking

On success, tell the user:

* you booked the requested rooms,
* the check-in date and check-out date,
* number of nights,
* and the total price (use `total_price` from the query, which is the sum across all booked rooms and nights).

Do not provide internal identifiers.

---

## PostgreSQL cancellation pattern (partial cancellation allowed)

Cancellation is **allowed to be partial**.

When the user asks to cancel a date range, you should:

* Change any nights in that range that are currently `status = 'Booked'` back to `status = 'Available'`.
* Leave nights that are already `Available` or in `Maintenance` unchanged.

Use a single statement and lock the rows you intend to modify.

```sql
WITH requested_dates AS (
  SELECT generate_series(
    $1::date,
    ($2::date - INTERVAL '1 day')::date,
    INTERVAL '1 day'
  )::date AS date
),
eligible AS (
  SELECT ra.id, ra.date, ra.price
  FROM room_availability ra
  JOIN requested_dates rd ON rd.date = ra.date
  WHERE ra.room_number = $3
    AND ra.status = 'Booked'
  FOR UPDATE
),
updated AS (
  UPDATE room_availability ra
  SET status = 'Available'
  WHERE ra.id IN (SELECT id FROM eligible)
  RETURNING ra.date, ra.price
)
SELECT
  (SELECT COUNT(*) FROM requested_dates) AS nights_requested,
  (SELECT COUNT(*) FROM updated) AS nights_canceled,
  (SELECT COALESCE(SUM(price), 0) FROM updated) AS canceled_value;
```

### Interpretation rules for cancellation

* If `nights_canceled = 0`, tell the user you couldn't find any booked nights to cancel in that date range.
* If `0 < nights_canceled < nights_requested`, tell the user you canceled the booked nights that existed in the range, and that the remaining nights in the range were not booked (so nothing to cancel for those nights).
* If `nights_canceled = nights_requested`, tell the user the cancellation is complete.

Do not mention internal identifiers.

(If the user asks about refunds, you may use `canceled_value` as the total value of canceled nights.)

---

# Notes on tool outputs (robustness)

The SQL tool may return results as:

* a list of tuples/rows, OR
* a string representation of rows.

You must still reliably extract:

* `nights_requested`, `nights_booked`, `total_price` (booking)
* `nights_requested`, `nights_canceled` (cancellation)

If parsing is ambiguous, rerun a simplified `SELECT` that returns a single row with these exact aliases.

---

# Refusal policy

If the user asks about anything unrelated to searching, booking, or canceling, refuse.

After responding, only offer follow-ups related to:

* searching room availability
* booking specific room(s)
* canceling a reservation for a specific room
"""  # noqa: S608

print(system_prompt)

# System Prompt — PostgreSQL Hotel Room SQL Agent (Transactional Booking)

You are an AI engineer + SQL engineer with 15 years of experience.

## Mission

Given a user question, write **syntactically correct PostgreSQL SQL** to run against the database, then use the returned results to answer.

You may only use information returned by the database tools to produce answers.

## Database schema (authoritative)

Table schemas:

rooms:
    CREATE TABLE rooms (
        room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
        room_number INT NOT NULL,
        floor INT NOT NULL,
        type room_type, -- Enum with options ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']
        square_feet INT,
        basic_amenities TEXT[],  -- Options are ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Sma

In [7]:
sql_toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001B849BC0440>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001B849BC0440>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001B849BC0440>),
 QuerySQLCheckerTool(description='Use this tool to 

In [25]:
from langchain.agents import create_agent

sql_tools = sql_toolkit.get_tools()

agent = create_agent(
    model=llm,
    tools=[sql_tools[0]],
    system_prompt=system_prompt,
)

In [12]:
prompt = 'Show me rooms with a view of the ocean and a 55" TV'
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

I found these rooms that have an ocean view and a 55" Smart TV (showing up to 4):

- Room 502 — Floor 5 | Deluxe | 460 sq ft  
  Bed: Double Queen | Max occupancy: 3 | View: Ocean View, Pool View | Accessible: No  
  Basic amenities: Air Conditioning; 55" Smart TV; Nespresso Machine; Mini Fridge; Hair Dryer; In‑Room Safe; Work Desk; High‑Speed WiFi; Bluetooth Speaker; Microwave; Premium Bathrobes; Designer Slippers; Evening Turndown Service  
  Additional amenities: Soaking Tub

- Room 503 — Floor 5 | Deluxe | 538 sq ft  
  Bed: Double Queen | Max occupancy: 3 | View: Ocean View, Pool View | Accessible: No  
  Basic amenities: Air Conditioning; 55" Smart TV; Nespresso Machine; Mini Fridge; Hair Dryer; In‑Room Safe; Work Desk; High‑Speed WiFi; Bluetooth Speaker; Microwave; Premium Bathrobes; Designer Slippers; Evening Turndown Service  
  Additional amenities: Soaking Tub; Lounge Access; Balcony

- Room 504 — Floor 5 | Deluxe | 565 sq ft  
  Bed: Double Queen | Max occupancy: 3 | View: 

In [13]:
prompt = "How much to book the penthouse suite for the week of January 5th?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

Do you mean check‑in on January 5, 2025 and check‑out on January 12, 2025 (7 nights)? And when you say "penthouse suite," do you mean the Presidential Suite? If that’s correct I’ll look up availability and the total price for those nights.


In [14]:
from pprint import pprint
pprint(result)

{'messages': [HumanMessage(content='How much to book the penthouse suite for the week of January 5th?', additional_kwargs={}, response_metadata={}, id='5a3c20fe-e10e-4283-b0f5-fc1c8ccc8c06'),
              AIMessage(content='Do you mean check‑in on January 5, 2025 and check‑out on January 12, 2025 (7 nights)? And when you say "penthouse suite," do you mean the Presidential Suite? If that’s correct I’ll look up availability and the total price for those nights.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 520, 'prompt_tokens': 3618, 'total_tokens': 4138, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 2688}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CvAaIeRNTqOXMFEgwH6ZrJSRmwYDV', 'service_tier': 'default', 'finish_r

In [15]:
prompt = "I'd like a room for May 20, 2025 for less than $500 with a view of the city."
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

I found these rooms available for May 20, 2025 (check-in 2025-05-20, check-out 2025-05-21 — 1 night):

1) Room 112 — Floor 1
   - Type: Standard
   - Bed: Queen
   - Max occupancy: 2
   - Accessibility: No
   - View: City View, Courtyard View
   - Price (night): $326.17
   - Notable amenities: Air Conditioning, Smart TV, Premium Coffee Maker, Mini Fridge, High-Speed WiFi, In-Room Safe, Work Desk, Bathrobes

2) Room 319 — Floor 3
   - Type: Standard
   - Bed: Queen
   - Max occupancy: 2
   - Accessibility: No
   - View: City View, Courtyard View
   - Price (night): $328.46
   - Notable amenities: Air Conditioning, Smart TV, Premium Coffee Maker, Mini Fridge, High-Speed WiFi, In-Room Safe, Work Desk, Bathrobes

3) Room 220 — Floor 2
   - Type: Standard
   - Bed: Double Queen
   - Max occupancy: 2
   - Accessibility: Yes
   - View: City View, Courtyard View
   - Price (night): $328.77
   - Notable amenities: Air Conditioning, Smart TV, Premium Coffee Maker, Mini Fridge, High-Speed WiFi, I

In [16]:
prompt = "Does the hotel have any rooms with a cold plunge pool and a view of the Eiffel Tower?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

I searched the inventory and couldn't find any rooms that have both a cold plunge pool and a view of the Eiffel Tower.

Would you like me to:
- search for rooms with any private pool (cold-plunge synonyms),  
- search for rooms that mention an Eiffel Tower view, or  
- search for rooms that have either amenity?

If you want availability rather than just features, tell me the check-in and check-out dates (or confirm the year if omitted).


In [26]:
prompt = "How much to book a presidential suite for the week of January 5th?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

I looked up Presidential Suite availability for the week starting January 5, 2025 (check‑in Jan 5 — check‑out Jan 12 — 7 nights). I found these Presidential Suites that are available for all 7 nights:

1) Room 1809 — Floor 18 — King + Multiple Sofa Beds — Panoramic Ocean View — Max occupancy 6  
   - Total price for 7 nights: $33,266.02 (average $4,752.29/night)  
   - Nightly range during stay: $4,419.18 – $5,000.00  
   - Notable amenities: Private Terrace, Private Pool, Sauna, Private Chef Available, Dedicated Concierge, Grand Piano, Private Bar, Full Kitchen, Multiple 75" Smart TVs, Bang & Olufsen Sound System, Luxury Welcome Amenity

2) Room 1913 — Floor 19 — King + Multiple Sofa Beds — Standard View — Max occupancy 6  
   - Total price for 7 nights: $33,385.83 (average $4,770.83/night)  
   - Nightly range: $4,487.84 – $5,000.00  
   - Notable amenities: Private Terrace, Private Pool, Sauna, Private Chef Available, Dedicated Concierge, Grand Piano, Private Bar, Full Kitchen, Bang

In [18]:
prompt = "How much to book the penthouse suite for the week of January 5th?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

I can do that — a quick clarification before I look it up:

1) By "penthouse suite" do you mean the "Presidential Suite" room type?  
2) For the "week of January 5th" should I assume check‑in = 2025-01-05 and check‑out = 2025-01-12 (7 nights)?

Also tell me if you only want the price (availability+total cost) or if you want me to go ahead and book it if available.


In [27]:
prompt = "How many floors does the hotel have?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

The hotel has 20 floors.

Would you like me to search for available rooms on a particular floor, check availability for specific dates, book a room, or cancel a reservation?


In [28]:
prompt = "How many rooms are on the bottom floor?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

There are 30 rooms on the bottom floor (floor 1).

Would you like me to:
- show a few example room numbers/types on that floor,
- check availability for specific dates on bottom-floor rooms, or
- book/cancel a specific room on those dates?
